In [1]:
import logging
import geopandas as gpd
from pathlib import Path
from functools import partial
import gc
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
logger = logging.getLogger("accessibility_pipeline")


In [2]:
def city_name_to_slug(city_name: str) -> str:
    return _city_slug(city_name) or "city"


In [3]:
# This is where I will define cities and categories
cell_size = 100  # meters (edge-to-edge spacing concept)
COUNTRY = "canada"
CITY_POLYGONS_RELATIVE = Path(f"data/processed/{COUNTRY}_cities.gpkg")
COUNTRY_CONFIG_RELATIVE = Path(f"config/{COUNTRY}.yaml")
ACCESSIBILITY_OUTPUT_RELATIVE = Path("data/processed/accessibility")
FRONTEND_PUBLIC_RELATIVE = Path("frontend/public")
INCLUDE_NETWORK_DEBUG_COLUMNS_IN_EXPORT = True

OVERPASS_URLS = [
    "https://overpass-api.de/api",
    "https://overpass.kumi.systems/api",
    "https://lz4.overpass-api.de/api",
]
OVERPASS_REQUEST_TIMEOUT_SECONDS = 300
OVERPASS_MAX_RETRIES_PER_ENDPOINT = 2
OVERPASS_BACKOFF_BASE_SECONDS = 5
OSMNX_NETWORK_TYPE = "walk"
OSMNX_RETAIN_ALL_COMPONENTS = True
OSMNX_TRUNCATE_BY_EDGE = True

categories = [
    {
        "id": "grocery",
        "usePolygons": False,
        "tags": {
            "shop": [
                "supermarket",
                "grocery",
                "greengrocer"
            ]
        }
    },

    {
        "id": "healthcare",
        "usePolygons": False,
        "tags": {
            "amenity": [
                "hospital",
                "clinic",
                "doctors",
                "pharmacy"
            ],
            "healthcare": [
                "clinic",
                "doctor",
                "hospital"
            ]
        }
    },

    {
        "id": "education",
        "usePolygons": True,
        "tags": {
            "amenity": [
                "school",
                "university",
                "college"
            ]
        }
    },
    {
        "id": "transit",
        "usePolygons": False,
        "tags": {
            "amenity": [
                "bus_station",
                "ferry_terminal"
            ],
            "public_transport": [
                "station",
                "stop_position"
            ],
            "railway": [
                "station",
                "tram_stop",
                "subway_entrance"
            ]
        }
    },
    {
        "id": "libraries",
        "usePolygons": False,
        "tags": {
            "amenity": [
                "library"
            ]
        }
    },

    {
        "id": "parks",
        "usePolygons": True,
        "tags": {
            "leisure": [
                "park",
                "garden",
                "nature_reserve",
                "playground"
            ],
            "landuse": [
                "recreation_ground"
            ]
        }
    }
]

In [4]:
import sys
# Ensure `src/` is importable regardless of notebook launch directory.
for parent in [Path.cwd(), *Path.cwd().parents]:
    src_dir = parent / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
from walkability import (
    compute_accessibility as _compute_accessibility,
    create_hex_grid as _create_hex_grid,
    export_results as _export_results,
    load_city as _load_city,
    load_pois as _load_pois,
)
from walkability.boundaries import city_slug as _city_slug
from walkability.utils import (
    build_accessibility_debug_summary,
    configure_osmnx_network_settings as _configure_osmnx_network_settings,
    create_points_along_line,
    fetch_features_with_retry as _fetch_features_with_retry,
    load_configured_cities as _load_configured_cities,
    resolve_output_dir,
    resolve_project_file,
)


In [5]:
def resolve_city_polygons_file():
    """
    Resolve the city polygons file from common notebook execution locations.
    """
    return resolve_project_file(CITY_POLYGONS_RELATIVE)



In [6]:
load_configured_cities = partial(
    _load_configured_cities,
    COUNTRY_CONFIG_RELATIVE,
    country_name=COUNTRY,
    logger=logger,
)


In [7]:
cities = load_configured_cities()

2026-06-25 10:20:27,027 INFO Using Canada config file: C:\Users\Art\Desktop\git\urban-accessibility-analysis\config\canada.yaml
2026-06-25 10:20:27,029 INFO Loaded 1 configured cities from Canada YAML


In [9]:
configure_osmnx_network_settings = partial(
    _configure_osmnx_network_settings,
    OVERPASS_REQUEST_TIMEOUT_SECONDS,
    overpass_rate_limit=True,
)


In [10]:
fetch_features_with_retry = partial(
    _fetch_features_with_retry,
    overpass_urls=OVERPASS_URLS,
    max_retries_per_endpoint=OVERPASS_MAX_RETRIES_PER_ENDPOINT,
    backoff_base_seconds=OVERPASS_BACKOFF_BASE_SECONDS,
    logger=logger,
)


In [11]:
load_city = partial(
    _load_city,
    network_type=OSMNX_NETWORK_TYPE,
    retain_all_components=OSMNX_RETAIN_ALL_COMPONENTS,
    truncate_by_edge=OSMNX_TRUNCATE_BY_EDGE,
    logger=logger,
)


In [12]:
create_hex_grid = _create_hex_grid

In [13]:
# `create_points_along_line` is now imported from walkability.utils.geometry.

In [14]:
load_pois = partial(
    _load_pois,
    fetch_features_fn=fetch_features_with_retry,
    create_points_along_line_fn=create_points_along_line,
    logger=logger,
)


In [15]:
compute_accessibility = partial(
    _compute_accessibility,
    logger=logger,
    walking_speed_m_per_min=72,
)


In [16]:
export_results = partial(
    _export_results,
    include_network_debug_columns_in_export=INCLUDE_NETWORK_DEBUG_COLUMNS_IN_EXPORT,
    accessibility_output_relative=ACCESSIBILITY_OUTPUT_RELATIVE,
    frontend_public_relative=FRONTEND_PUBLIC_RELATIVE,
    city_slug_fn=city_name_to_slug,
    resolve_output_dir_fn=resolve_output_dir,
    resolve_project_file_fn=resolve_project_file,
    logger=logger,
)


In [17]:
# `build_accessibility_debug_summary` is now imported from walkability.utils.diagnostics.

In [18]:
# Load city polygons file ONCE before processing all cities
city_polygons_file = resolve_city_polygons_file()
logger.info(f"Loading city polygons file: {city_polygons_file}")
city_polygons = gpd.read_file(city_polygons_file)
logger.info(f"City polygons loaded: {len(city_polygons)} features")


2026-06-25 10:20:27,120 INFO Loading city polygons file: C:\Users\Art\Desktop\git\urban-accessibility-analysis\data\processed\canada_cities.gpkg
2026-06-25 10:20:27,176 INFO City polygons loaded: 6 features


In [19]:
for city in cities:
    logger.info(f"\n{'='*60}")
    logger.info(f"Processing city: {city}")
    logger.info(f"{'='*60}")

    configure_osmnx_network_settings()

    try:
        boundary, G, target_crs, city_polygon_wgs84 = load_city(city, city_polygons)
    except LookupError as exc:
        logger.warning(f"Skipping city '{city}': {exc}")
        continue

    logger.info(f"Creating hex grid for boundary with cell size {cell_size} meters")
    grid = create_hex_grid(boundary, cell_size, target_crs)
    logger.info("Hex grid created and centroids computed")

    for category in categories:
        logger.info(f"Starting category: {category['id']}")

        pois = load_pois(city_polygon_wgs84, category, target_crs)

        grid = compute_accessibility(
            grid=grid,
            graph=G,
            pois=pois,
            category_id=category["id"],
            target_crs=target_crs
        )

    debug_summary = build_accessibility_debug_summary(grid, categories)
    logger.info(f"Accessibility debug summary for {city}:")
    print(debug_summary.to_string(index=False))

    output_dir = resolve_output_dir(ACCESSIBILITY_OUTPUT_RELATIVE)
    city_slug = city_name_to_slug(city)
    debug_summary_path = output_dir / f"{city_slug}_accessibility_debug_summary.csv"
    debug_summary.to_csv(debug_summary_path, index=False)
    logger.info(f"Saved debug summary: {debug_summary_path}")

    export_results(city, grid, G)

    # Explicitly delete large objects to free memory before processing next city
    del boundary, G, target_crs, city_polygon_wgs84, grid, debug_summary
    gc.collect()
    logger.info(f"Memory cleaned up after {city}")



2026-06-25 10:20:27,187 INFO 
2026-06-25 10:20:27,188 INFO Processing city: Regina
2026-06-25 10:20:27,188 INFO ============================================================
2026-06-25 10:20:27,189 INFO Loading city data for Regina
2026-06-25 10:20:37,674 INFO Network diagnostics: components=111, largest_component_nodes=27142, retain_all=True, truncate_by_edge=True
2026-06-25 10:20:42,335 INFO City boundary cleaned by removing water areas
2026-06-25 10:20:42,337 INFO Creating hex grid for boundary with cell size 100 meters
2026-06-25 10:20:43,169 INFO Hex grid created and centroids computed
2026-06-25 10:20:43,170 INFO Starting category: grocery
2026-06-25 10:20:43,170 INFO Loading POIs for category 'grocery'
2026-06-25 10:20:43,170 INFO POI fetch 'grocery' via https://overpass-api.de/api (attempt 1/2)
2026-06-25 10:20:46,675 INFO Loaded 20 point POIs for category 'grocery'
2026-06-25 10:20:46,676 INFO Computing accessibility for 'grocery'
2026-06-25 10:20:48,442 INFO Using 20 unique so

  category  hex_count  valid_hexes  zero_min_hexes  zero_min_pct  source_node_hexes  zero_min_and_source_hexes  zero_min_and_source_pct_of_zero  median_walk_m  median_snap_m  max_source_poi_count_on_node
 education       8733         8283             185          2.12                185                        185                            100.0        1220.83         110.87                             7
   grocery       8733         8283               1          0.01                  1                          1                            100.0        2943.12         110.87                             1
healthcare       8733         8283               7          0.08                  7                          7                            100.0        2249.10         110.87                             1
 libraries       8733         8284               0          0.00                  0                          0                              0.0        4601.96         110.87           

2026-06-25 10:24:46,794 INFO Created 8,733 records
2026-06-25 10:24:46,798 INFO GeoJSON written: C:\Users\Art\Desktop\git\urban-accessibility-analysis\data\processed\accessibility\regina_accessibility.geojson
2026-06-25 10:24:46,992 INFO Gzipped GeoJSON written: C:\Users\Art\Desktop\git\urban-accessibility-analysis\data\processed\accessibility\regina_accessibility.geojson.gz
2026-06-25 10:24:50,363 INFO Created 80,482 records
2026-06-25 10:24:50,390 INFO Network edges GeoJSON written: C:\Users\Art\Desktop\git\urban-accessibility-analysis\data\processed\accessibility\regina_network_edges.geojson
2026-06-25 10:24:50,772 INFO Created 27,457 records
2026-06-25 10:24:50,781 INFO Network nodes GeoJSON written: C:\Users\Art\Desktop\git\urban-accessibility-analysis\data\processed\accessibility\regina_network_nodes.geojson
2026-06-25 10:24:50,789 INFO Copied GeoJSON to frontend: C:\Users\Art\Desktop\git\urban-accessibility-analysis\frontend\public\regina_accessibility.geojson
2026-06-25 10:24:5